In [1]:
!pip install duckdb FlagEmbedding pymupdf4llm httpx pdfplumber

In [2]:
import duckdb
from FlagEmbedding import BGEM3FlagModel
import torch
import numpy as np
import httpx
from pathlib import Path
import pymupdf4llm
import re

con = duckdb.connect()
con.install_extension("vss")
con.load_extension("vss")

sql="""SET GLOBAL hnsw_enable_experimental_persistence = true;"""

con.execute(sql)


In [3]:
if not Path('handbook.pdf').exists():
    pdfreq = httpx.get('https://www.501commons.org/resources/tools-and-best-practices/human-resources/sample-employee-handbook-national-council-of-nonprofits', verify=False)
    pdf = pdfreq.content
    with open('handbook.pdf', 'wb') as f:
        f.write(pdf)

In [4]:
# `ignore_graphics` keeps it from interpreting random lines as tables.
handbook = pymupdf4llm.to_markdown('handbook.pdf', table_strategy='lines_strict', ignore_graphics=True)

In [5]:
handbook = handbook.split('I. MISSION')[2]

In [6]:
handbook = handbook.split('\n\n')

In [7]:
section = ''
subsection = ''
paragraphs = []

for para in handbook[:100]:
    para = para.strip()
    if re.match(r'[IVX]+\. [\w]+', para):
        para = re.sub(r'[IVX]+\. ', '', para)
        section = para
        subsection = ''
    elif re.match(r'[A-Z]\. [\w]+', para):
        para = re.sub(r'[A-Z]\. ', '', para)
        subsection = para
    elif re.match(r'\[\d+\s?\]', para): # Page numbers. ignore.
        ...
    elif para.strip() == '':
        ...
    else:
        paragraphs.append({
            'section': section,
            'subsection': subsection,
            'paragraph': para
        })

In [8]:
for i in range(len(paragraphs)):
    paragraphs[i]['paragraph'] = re.sub(r'\s+', ' ', paragraphs[i]['paragraph'])
    paragraphs[i]['context'] = paragraphs[i]['paragraph']
    if i > 0 and\
      paragraphs[i]['section'] == paragraphs[i-1]['section'] and\
      paragraphs[i]['subsection'] == paragraphs[i-1]['subsection']:
        paragraphs[i-1]['paragraph'] = re.sub(r'\s+', ' ', paragraphs[i-1]['paragraph'])
        paragraphs[i]['context'] = paragraphs[i-1]['paragraph'] + '\n\n' + paragraphs[i]['context']

    if i < len(paragraphs)-1 and\
      paragraphs[i]['section'] == paragraphs[i+1]['section'] and\
      paragraphs[i]['subsection'] == paragraphs[i+1]['subsection']:
        paragraphs[i+1]['paragraph'] = re.sub(r'\s+', ' ', paragraphs[i+1]['paragraph'])
        paragraphs[i]['context'] = paragraphs[i]['context'] + '\n\n' + paragraphs[i+1]['paragraph']

In [9]:
for i in range(10):
  print(paragraphs[i])

{'section': 'OVERVIEW', 'subsection': '', 'paragraph': 'The {ORGANIZATION NAME} Employee Handbook (the “Handbook”) has been developed to provide general guidelines about {ORGANIZATION NAME} policies and procedures for employees. It is a guide to assist you in becoming familiar with some of the privileges and obligations of your employment, including {ORGANIZATION NAME}ʹs policy of voluntary at‐will employment. None of the policies or guidelines in the Handbook are intended to give rise to contractual rights or obligations, or to be construed as a guarantee of employment for any specific period of time, or any specific type of work. Additionally, with the exception of the voluntary at‐will employment policy, these guidelines are subject to modification, amendment or revocation by {ORGANIZATION NAME} at any time, without advance notice.', 'context': 'The {ORGANIZATION NAME} Employee Handbook (the “Handbook”) has been developed to provide general guidelines about {ORGANIZATION NAME} polic

In [10]:
device = "cpu"
# use a GPU if available to speed up the embedding computation
if torch.cuda.is_available(): device = "cuda" # Nvidia GPU
elif torch.backends.mps.is_available(): device = "mps" # Apple silicon GPU

model = BGEM3FlagModel('BAAI/bge-m3', use_fp16=True, device=device)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Fetching 30 files:   0%|          | 0/30 [00:00<?, ?it/s]

In [11]:
con.execute("CREATE SEQUENCE IF NOT EXISTS seq_handbookid START 1")
con.execute("DROP TABLE IF EXISTS handbook")
qry = """CREATE TABLE handbook
(
  id INTEGER PRIMARY KEY DEFAULT NEXTVAL('seq_handbookid'),
  section TEXT,
  subsection TEXT,
  context TEXT,
  embedding FLOAT[1024]
)

"""

con.execute(qry)

In [12]:
for para in paragraphs:
    embedding = model.encode(para['paragraph'])["dense_vecs"]
    qry = f"""INSERT INTO handbook (section, subsection, context, embedding) VALUES (?, ?, ?, ?)"""
    con.execute(qry, (para['section'], para['subsection'], para['context'], embedding))

You're using a XLMRobertaTokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.


In [13]:
con.execute('SELECT * FROM handbook LIMIT 5').fetch_df()

,id,section,subsection,context,embedding
0,1,OVERVIEW,,The {ORGANIZATION NAME} Employee Handbook (the...,"[-0.025878906, -0.01033783, -0.051971436, -0.0..."
1,2,OVERVIEW,,The {ORGANIZATION NAME} Employee Handbook (the...,"[0.0021438599, -0.04321289, -0.03894043, 0.004..."
2,3,OVERVIEW,,The personnel polices of {ORGANIZATION NAME} a...,"[-0.0063285828, -0.012916565, -0.059173584, -0..."
3,4,VOLUNTARY AT‐WILL EMPLOYMENT,,Unless an employee has a written employment ag...,"[-0.011520386, 0.0011253357, -0.01486969, 0.00..."
4,5,EQUAL EMPLOYMENT OPPORTUNITY,,{ORGANIZATION NAME} shall follow the spirit an...,"[-0.047790527, -0.022766113, -0.021026611, 0.0..."


In [14]:
from duckdb.typing import VARCHAR

def embed(sentence: str) -> np.ndarray:
    return model.encode(sentence)['dense_vecs']

con.create_function("embed", embed, [VARCHAR], 'FLOAT[1024]')


qry = "SELECT embed('How much can I drink at work?') AS query_embedding;"
con.execute(qry).fetch_df()

,query_embedding
0,"[-0.017196655, 0.012336731, -0.041107178, -0.0..."


In [15]:
def search(q: str):
    return con.execute("""
        FROM handbook
        SELECT section, subsection, context, array_inner_product(embedding, embed($q)) AS similarity
        ORDER BY similarity DESC
        LIMIT 5""",
        {"q": q}
    ).pl()

search('Do you reimburse travel expenses?')

section,subsection,context,similarity
str,str,str,f32
"""LEAVE BENEFITS AND OTHER WORK …","""Vacation""","""Full‐time employees will conti…",0.494798
"""POSITION DESCRIPTION AND SALAR…","""""","""qualifications required, salar…",0.49229
"""LEAVE BENEFITS AND OTHER WORK …","""Vacation""","""During the first 90 days of em…",0.48713
"""LEAVE BENEFITS AND OTHER WORK …","""Vacation""","""During the first 90 days of em…",0.475331
"""LEAVE BENEFITS AND OTHER WORK …","""Sick Leave""","""Sick leave benefits are earned…",0.472641
